In [62]:
import pandas as pd
import re

df=pd.read_csv("train.csv")

mots_inutiles = set([
    "i","me","my","myself","we","our","ours","ourselves",
    "you","your","yours","yourself","yourselves",
    "he","him","his","himself",
    "she","her","hers","herself",
    "it","its","itself",
    "they","them","their","theirs","themselves",
    "what","which","who","whom","this","that","these","those",
    "am","is","are","was","were","be","been","being",
    "have","has","had","do","does","did",
    "a","an","the","and","or","but","if","because","as",
    "until","while","of","at","by","for","with","about","against",
    "between","into","through","during","before","after","above","below",
    "to","from","up","down","in","out","on","off","over","under",
    "again","further","then","once","get", "just", "like", "can", "not","know",
    "want","time","how","when","now","all","feel", "really", "things", "just",
    "like","very", "too", "lot", "there", "back",
    "even", "every", "years", "told",
    "going", "still", "one", "now","will"
])
contractions = {
    "don't":"do not",
    "can't":"can not",
    "i'm":"i am",
    "it's":"it is",
    "i've":"i have",
    "you're":"you are",
    "we're":"we are",
    "they're":"they are"
}
def preprocess(texte):
    if pd.isna(texte):
        return []

    texte=texte.lower()
    for k,v in contractions.items():
        texte=texte.replace(k, v)
    texte=re.sub(r'[^\w\s]',' ',texte)

    tokens=texte.split()

    tokens=[t for t in tokens if t not in mots_inutiles]
    tokens=[t for t in tokens if len(t) > 2]

    return tokens

df['patient_clean'] = df.iloc[:,0].apply(preprocess)
df['therapist_clean'] = df.iloc[:,1].apply(preprocess)
nb_patients=df.iloc[:,0].nunique()
print("Patients uniques:", nb_patients)
nb_therapeutes=df.iloc[:,1].nunique()
print("Thérapeutes uniques:", nb_therapeutes)


echange_moyens = len(df) / nb_patients
print("Échanges moyens par patient:", echange_moyens)



Patients uniques: 995
Thérapeutes uniques: 2479
Échanges moyens par patient: 3.52964824120603


In [63]:
from collections import Counter

tout_mots=[mot for tokens in df['patient_clean'] for mot in tokens]
freq = Counter(tout_mots)

print(freq.most_common(20))

[('never', 623), ('always', 509), ('relationship', 504), ('should', 461), ('think', 442), ('people', 434), ('love', 428), ('anxiety', 412), ('other', 406), ('boyfriend', 396), ('help', 392), ('counseling', 390), ('life', 377), ('having', 353), ('issues', 344), ('more', 331), ('need', 331), ('sex', 330), ('family', 326), ('depression', 320)]


In [64]:
pip install sentence-transformers

In [65]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

df['patient_text'] = df['patient_clean'].apply(lambda x: " ".join(x))

df = df.drop_duplicates(subset="patient_text")
df = df.drop_duplicates(subset="patient_clean")

tfidf = TfidfVectorizer(max_features=2000)

X = tfidf.fit_transform(df['patient_text'])
kmeans = KMeans(n_clusters=8, random_state=42)
df['cluster_tfidf'] = kmeans.fit_predict(X)
print(df[['patient_text', 'cluster_tfidf']].head(10))
print(df['cluster_tfidf'].value_counts())
print(df[df['cluster_tfidf'] == 0]['patient_text'].head(10))
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
terms = tfidf.get_feature_names_out()

for i in range(8):
    print(f"Cluster {i}")
    top_words = [terms[ind] for ind in order_centroids[i, :10]]
    print(top_words)

                                         patient_text  cluster_tfidf
0   some feelings barely sleep nothing think worth...              2
23  many issues address history sexual abuse breas...              6
70  feeling more more month started having trouble...              1
72  facing severe depression anxiety distracts can...              6
81                        place where content day day              3
84  severe problem major several minor operations ...              6
86  suffer adult adhd anxiety disorder depression ...              0
89  few ago making love wife known reason lost ere...              0
95  struggle depression well pretty intense mood s...              1
99  self harm stop awhile see something sad depres...              5
cluster_tfidf
6    197
1    138
5     97
0     91
2     83
3     82
7     77
4     66
Name: count, dtype: int64
86     suffer adult adhd anxiety disorder depression ...
89     few ago making love wife known reason lost ere...
146    dealing 

In [66]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
words = ["anxiety", "depression", "stress", "sleep"]
word_embeddings = embed_model.encode(words)
sim_matrix = cosine_similarity(word_embeddings)

all_words = list(set(sum(df['patient_clean'], [])))

embeddings = embed_model.encode(all_words)

def most_similar(word, words, embeddings, top_n=10):
    import numpy as np
    from sklearn.metrics.pairwise import cosine_similarity
    idx=words.index(word)
    sims=cosine_similarity([embeddings[idx]], embeddings)[0]
    top_idx=sims.argsort()[::-1][1:top_n+1]
    return [(words[i], round(float(sims[i]), 3))
        for i in top_idx]

result=most_similar("depression", all_words, embeddings)
pd.DataFrame(result, columns=["word", "similarity"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,word,similarity
0,depressive,0.885
1,depressed,0.861
2,psychiatric,0.703
3,suicide,0.661
4,depressing,0.661
5,depressants,0.638
6,misery,0.631
7,happiness,0.627
8,unhappy,0.616
9,suicidal,0.611


In [67]:
result2=most_similar("anxiety", all_words, embeddings)
pd.DataFrame(result2, columns=["word", "similarity"])

,word,similarity
0,anxious,0.855
1,anxieties,0.715
2,nervous,0.674
3,fear,0.669
4,fears,0.663
5,terrified,0.624
6,panic,0.620
7,psychiatric,0.601
8,paranoid,0.600
9,uncomfortable,0.592


In [68]:
result3=most_similar("stress", all_words, embeddings)
pd.DataFrame(result3, columns=["word", "similarity"])

,word,similarity
0,stressing,0.929
1,stressed,0.928
2,stressors,0.876
3,stresses,0.862
4,stressor,0.836
5,stressful,0.792
6,tension,0.639
7,anxiety,0.586
8,anxious,0.585
9,frustration,0.570


In [69]:
result4=most_similar("sleep", all_words, embeddings)
pd.DataFrame(result4, columns=["word", "similarity"])

,word,similarity
0,sleeping,0.906
1,sleeps,0.894
2,asleep,0.806
3,slept,0.786
4,nap,0.761
5,waking,0.660
6,dreaming,0.641
7,bed,0.637
8,wakes,0.635
9,awake,0.619


In [70]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
vectorizer = CountVectorizer(max_features=2000)
X = vectorizer.fit_transform(df['patient_text'])


In [71]:
lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X)
mots=vectorizer.get_feature_names_out()

for i, sujet in enumerate(lda.components_):
    print(f"Sujet {i}")
    print([mots[i] for i in sujet.argsort()[-10:]])

Sujet 0
['day', 'three', 'anxiety', 'people', 'friends', 'think', 'doesn', 'help', 'disorder', 'family']
Sujet 1
['school', 'friend', 'day', 'home', 'never', 'family', 'together', 'boyfriend', 'said', 'love']
Sujet 2
['away', 'life', 'girlfriend', 'never', 'sex', 'boyfriend', 'always', 'relationship', 'mom', 'other']
Sujet 3
['talk', 'having', 'make', 'help', 'always', 'love', 'think', 'life', 'boyfriend', 'never']
Sujet 4
['more', 'relationship', 'see', 'sex', 'says', 'husband', 'always', 'should', 'think', 'past']
